# TourLingo AI - Neural Machine Translation (NMT) Fine-Tuning
**Project ID:** CSEAIML2026-P26  
**Dataset:** IIT Bombay English-Hindi Parallel Corpus (`cfilt/iitb-english-hindi`)  
**Base Architecture:** Helsinki-NLP MarianMT (`opus-mt-en-hi`)

This notebook trains and fine-tunes a Sequence-to-Sequence NMT model for tourist & conversational travel translation.

### Step 1: Install Required Packages

In [ ]:
!pip install -q transformers datasets evaluate accelerate sacrebleu sentencepiece torch

### Step 2: Load the IIT Bombay English-Hindi Dataset

In [ ]:
from datasets import load_dataset

print("Loading IIT Bombay Parallel Corpus...")
dataset = load_dataset("cfilt/iitb-english-hindi")

print(f"Total Training Pairs: {len(dataset['train']):,}")
print(f"Total Validation Pairs: {len(dataset['validation']):,}")

# Sample pair
sample = dataset["train"][0]["translation"]
print("Sample English:", sample["en"])
print("Sample Hindi:  ", sample["hi"])

### Step 3: Load Pretrained MarianMT Tokenizer & Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-hi"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device.upper()}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

### Step 4: Tokenize Dataset Slice (15,000 to 50,000 pairs)

In [ ]:
NUM_SAMPLES = 20000
MAX_LEN = 128

# Slice high quality subset for fast student training
train_data = dataset["train"].shuffle(seed=42).select(range(NUM_SAMPLES))
val_data = dataset["validation"].shuffle(seed=42).select(range(1000))

def preprocess_batch(examples):
    inputs = [ex["en"] if ex["en"] else "" for ex in examples["translation"]]
    targets = [ex["hi"] if ex["hi"] else "" for ex in examples["translation"]]
    
    model_inputs = tokenizer(inputs, max_length=MAX_LEN, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_data.map(preprocess_batch, batched=True, remove_columns=["translation"])
tokenized_val = val_data.map(preprocess_batch, batched=True, remove_columns=["translation"])

### Step 5: Configure Seq2Seq Trainer with SacreBLEU

In [ ]:
import evaluate
import numpy as np
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    result = bleu_metric.compute(predictions=[p.strip() for p in decoded_preds], references=[[l.strip()] for l in decoded_labels])
    return {"bleu": round(result["score"], 2)}

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="custom_en_hi_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=(device == "cuda"),
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting training...")
trainer.train()

### Step 6: Test on Tourist & Vendor Dialogues

In [ ]:
test_tourist_sentences = [
    "How much does this cost?",
    "Can you give me a discount?",
    "Where is the nearest railway station?",
    "Please make the food non-spicy.",
    "I need help immediately, call the police."
]

model.eval()
for s in test_tourist_sentences:
    inputs = tokenizer(s, return_tensors="pt", max_length=128, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=128, num_beams=4)
    translated = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"Tourist: '{s}' ➔ Vendor: '{translated}'")

### Step 7: Export Model for Local TourLingo App

In [ ]:
# Save locally and zip for downloading to your local 'models/custom_en_hi_model' folder
trainer.save_model("custom_en_hi_model")
tokenizer.save_pretrained("custom_en_hi_model")

!zip -r custom_en_hi_model.zip custom_en_hi_model
print("Model saved and zipped! You can now download custom_en_hi_model.zip and extract into your local nlp/models directory.")